# Phi-3 Mini QLoRA Fine-Tuning for Functional Skill Classification

## 1. Install Dependencies


In [1]:
!pip install -q -U transformers accelerate peft bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 27.3 MB/s eta 0:00:00


## 2. Verify GPU Availability


In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Please enable GPU by going to Settings on the right and setting Accelerator to GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory (GB):",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
)

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56


## 3. Import Libraries and Configure Global Settings

This section imports all required libraries and defines the global configuration used throughout the notebook.


In [3]:
import os
import glob
import random
import inspect
import warnings
import shutil

import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    set_seed,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

warnings.filterwarnings("ignore")

MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
SEED = 42
MAX_LENGTH = 512
MAX_DESCRIPTION_TOKENS = 400

# Use a small sample for initial testing. After confirming it works, increase the sample size or set it to None to process the full dataset.
TRAIN_SAMPLE_SIZE = 1000
VAL_SAMPLE_SIZE = 200
TEST_SAMPLE_SIZE = 200

LORA_RANK = 16
OUTPUT_DIR = "/kaggle/working/phi3_skill_lora"

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

print("Model:", MODEL_NAME)
print("Output:", OUTPUT_DIR)

Model: microsoft/Phi-3-mini-4k-instruct
Output: /kaggle/working/phi3_skill_lora


## 4. Explore the Kaggle Input Directory


In [4]:
for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files[:20]:
        print(f"{indent}  {file}")

input/
  datasets/
    hollychen12345/
      train-dataset/
        validation.xls
        train.xls
        test.xls


## 5. Locate and Load the Dataset

This notebook automatically searches for:

- train.csv (or train.xls containing CSV data)
- validation.csv
- test.csv

If multiple files are found, the first match will be used.


In [5]:
def find_file(possible_names):
    matches = []

    for name in possible_names:
        matches.extend(
            glob.glob(f"/kaggle/input/**/{name}", recursive=True)
        )

    matches = sorted(set(matches))

    if not matches:
        raise FileNotFoundError(
            f"Could not find any of the following files: {possible_names}\n"
            "Please make sure the dataset has been added using 'Add Input'."
        )

    if len(matches) > 1:
        print("Multiple matching files were found:")
        for path in matches:
            print("  -", path)
        print(f"Using the first match by default: {matches[0]}")

    return matches[0]


TRAIN_PATH = find_file(["train.csv", "train.xls"])
VAL_PATH = find_file([
    "validation.csv", "validation.xls", "val.csv", "val.xls"
])
TEST_PATH = find_file(["test.csv", "test.xls"])

print("TRAIN_PATH:", TRAIN_PATH)
print("VAL_PATH:", VAL_PATH)
print("TEST_PATH:", TEST_PATH)


def read_table(path):
    try:
        return pd.read_csv(path)
    except Exception as csv_error:
        try:
            return pd.read_excel(path)
        except Exception as excel_error:
            raise RuntimeError(
                f"can't read {path}\n"
                f"read_csv: {csv_error}\n"
                f"read_excel: {excel_error}"
            )


train_df = read_table(TRAIN_PATH)
val_df = read_table(VAL_PATH)
test_df = read_table(TEST_PATH)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

display(train_df.head())

TRAIN_PATH: /kaggle/input/datasets/hollychen12345/train-dataset/train.xls
VAL_PATH: /kaggle/input/datasets/hollychen12345/train-dataset/validation.xls
TEST_PATH: /kaggle/input/datasets/hollychen12345/train-dataset/test.xls
Train: (86224, 10)
Validation: (10778, 10)
Test: (10779, 10)


,job_id,company_name,title,description,location,formatted_work_type,formatted_experience_level,description_character_count,description_word_count,skill_name
0,3905354058,Connected Health Care,Registered Nurse - Intensive and Inpatient Car...,Job Description Join Our Team as an Intensive ...,"Napoleon, OH",Full-time,Mid-Senior level,1805,249,Health Care Provider
1,3887715508,MyEyeDr.,Optometrist (OD) - Part Time,Description MyEyeDr. is seeking a part-time Op...,"Katy, TX",Part-time,Entry level,3388,491,Health Care Provider
2,3887714023,ALTEN Technology USA,Test Engineer L1,"We’re ALTEN Technology USA, an engineering com...","Raymond, OH",Full-time,Entry level,3456,488,"Engineering, Information Technology"
3,3903448231,"Sunlite Plastics, Inc.",Mechanical Engineer,Sunlite Plastics is a custom extrusion company...,"Germantown, WI",Full-time,Mid-Senior level,3694,480,"Engineering, Information Technology"
4,3884836213,CRC Insurance Services,Account Executive,The position is described below. If you want t...,"Magnolia, TX",Full-time,Mid-Senior level,5279,740,"Business Development, Sales"


## 6. Validate the Dataset

Verify that the required columns exist:

- title
- description
- skill_name


In [6]:
REQUIRED_COLUMNS = ["title", "description", "skill_name"]

def validate_dataframe(df, name):
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(
            f"The {name} dataset is missing the following required columns: {missing}\n"
            f"Available columns: {list(df.columns)}"
        )

    print(name, "rows:", len(df))
    print("Missing title:", df["title"].isna().sum())
    print("Missing description:", df["description"].isna().sum())
    print("Missing skill_name:", df["skill_name"].isna().sum())


validate_dataframe(train_df, "train_df")
validate_dataframe(val_df, "val_df")
validate_dataframe(test_df, "test_df")

for df in [train_df, val_df, test_df]:
    df["title"] = df["title"].fillna("").astype(str)
    df["description"] = df["description"].fillna("").astype(str)
    df["skill_name"] = df["skill_name"].fillna("").astype(str)

train_df rows: 86224
Missing title: 0
Missing description: 0
Missing skill_name: 0
val_df rows: 10778
Missing title: 0
Missing description: 0
Missing skill_name: 0
test_df rows: 10779
Missing title: 0
Missing description: 0
Missing skill_name: 0


## 7. Create a Development Subset

A small subset is used first to verify that the complete training pipeline works correctly before training on the full dataset.


In [7]:
def sample_dataframe(df, sample_size):
    if sample_size is None or sample_size >= len(df):
        return df.reset_index(drop=True)
    return df.sample(
        n=sample_size,
        random_state=SEED
    ).reset_index(drop=True)


train_small = sample_dataframe(train_df, TRAIN_SAMPLE_SIZE)
val_small = sample_dataframe(val_df, VAL_SAMPLE_SIZE)
test_small = sample_dataframe(test_df, TEST_SAMPLE_SIZE)

print("Training rows:", len(train_small))
print("Validation rows:", len(val_small))
print("Test rows:", len(test_small))

Training rows: 1000
Validation rows: 200
Test rows: 200


## 8. Load the Phi-3 Tokenizer


In [8]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Pad token: <|endoftext|>
EOS token: <|endoftext|>


## 9. Build the Prompt Template


In [9]:
def build_user_prompt(title, description):
    title = str(title).strip()
    description = str(description).strip()

    return (
        "Classify the following job posting into one functional skill category.\n\n"
        f"Job Title:\n{title}\n\n"
        f"Job Description:\n{description}\n\n"
        "Return only the category name."
    )


def build_full_text(title, description, skill_name):
    user_prompt = build_user_prompt(
        title,
        description
    )

    messages = [
        {
            "role": "user",
            "content": user_prompt
        },
        {
            "role": "assistant",
            "content": str(skill_name).strip()
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

## 10. Preview a Training Example


In [10]:
example = train_small.iloc[0]

print(
    build_full_text(
        example["title"],
        example["description"][:1000],
        example["skill_name"],
    )
)

<|user|>
Classify the following job posting into one functional skill category.

Job Title:
CT Technologist *$20k Sign-on!*

Job Description:
Job Description Full Time CT Technologist in Aspen, Colorado Sign-On Bonus of up to $20,000! About The Position: Connected Health Care is thrilled to present a rewarding opportunity for a Full Time CT Technologist in beautiful Aspen, Colorado. We are partnering with a prestigious healthcare facility that values its team members and offers competitive compensation, including a potential sign-on bonus of up to $20,000! Job Description: As a Full Time CT Technologist, you will: Perform diagnostic CT scans on patients according to physician orders. Ensure patient safety and comfort during imaging procedures. Maintain and operate CT equipment, troubleshooting issues as needed. Collaborate with healthcare professionals to provide high-quality patient care. Adhere to protocols and safety standards. Qualifications: Graduate of an accredited Radiologic Te

## 11. Build the Custom PyTorch Dataset


In [11]:
class SkillExtractionDataset(Dataset):
    def __init__(
        self,
        dataframe,
        tokenizer,
        max_length=512,
        max_description_tokens=400,
    ):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.max_description_tokens = max_description_tokens

    def __len__(self):
        return len(self.df)

    def truncate_description(self, description):
        description_ids = self.tokenizer(
            str(description),
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_description_tokens,
        )["input_ids"]

        return self.tokenizer.decode(
            description_ids,
            skip_special_tokens=True,
        )

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        title = str(row["title"])
        description = self.truncate_description(row["description"])
        skill_name = str(row["skill_name"])

        user_prompt = build_user_prompt(title, description)

        prompt_messages = [
            {"role": "user", "content": user_prompt}
        ]

        full_messages = [
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": skill_name},
        ]

        prompt_text = self.tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        full_text = self.tokenizer.apply_chat_template(
            full_messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        encoded = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )

        prompt_encoded = self.tokenizer(
            prompt_text,
            truncation=True,
            max_length=self.max_length,
            add_special_tokens=False,
        )

        input_ids = encoded["input_ids"].squeeze(0)
        attention_mask = encoded["attention_mask"].squeeze(0)
        labels = input_ids.clone()

        prompt_length = min(
            len(prompt_encoded["input_ids"]),
            self.max_length,
        )

        labels[:prompt_length] = -100
        labels[attention_mask == 0] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

## 12. Verify Label Masking


In [12]:
train_dataset = SkillExtractionDataset(
    train_small,
    tokenizer,
    MAX_LENGTH,
    MAX_DESCRIPTION_TOKENS,
)

val_dataset = SkillExtractionDataset(
    val_small,
    tokenizer,
    MAX_LENGTH,
    MAX_DESCRIPTION_TOKENS,
)

test_dataset = SkillExtractionDataset(
    test_small,
    tokenizer,
    MAX_LENGTH,
    MAX_DESCRIPTION_TOKENS,
)

for i in range(min(5, len(train_dataset))):
    item = train_dataset[i]
    active_labels = item["labels"][item["labels"] != -100]

    print(f"Example {i}")
    print("Tokens used for loss:", len(active_labels))
    print(
        "Decoded labels:",
        tokenizer.decode(active_labels, skip_special_tokens=True)
    )
    print("-" * 60)

Example 0
Tokens used for loss: 6
Decoded labels: Health Care Provider
------------------------------------------------------------
Example 1
Tokens used for loss: 7
Decoded labels: Management, Manufacturing
------------------------------------------------------------
Example 2
Tokens used for loss: 8
Decoded labels: Engineering, Information Technology, Research
------------------------------------------------------------
Example 3
Tokens used for loss: 6
Decoded labels: Health Care Provider
------------------------------------------------------------
Example 4
Tokens used for loss: 7
Decoded labels: Information Technology, Project Management
------------------------------------------------------------


## 13. Configure 4-bit Quantization and LoRA


In [13]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "qkv_proj",
        "o_proj",
        "gate_up_proj",
        "down_proj",
    ],
)

print("QLoRA configuration ready.")

QLoRA configuration ready.


## 14. Load the Phi-3 Model


In [14]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

model.config.use_cache = False

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

trainable params: 25,165,824 || all params: 3,846,245,376 || trainable%: 0.6543


## 15. Configure the Hugging Face Trainer


In [15]:
training_kwargs = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": 2,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,
    "logging_steps": 10,
    "save_steps": 100,
    "eval_steps": 100,
    "save_total_limit": 2,
    "fp16": True,
    "bf16": False,
    "optim": "paged_adamw_8bit",
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.03,
    "weight_decay": 0.01,
    "report_to": "none",
    "remove_unused_columns": False,
    "gradient_checkpointing": True,
    "seed": SEED,
}

signature = inspect.signature(
    TrainingArguments.__init__
).parameters

if "eval_strategy" in signature:
    training_kwargs["eval_strategy"] = "steps"
elif "evaluation_strategy" in signature:
    training_kwargs["evaluation_strategy"] = "steps"

if "save_strategy" in signature:
    training_kwargs["save_strategy"] = "steps"

training_args = TrainingArguments(**training_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print(training_args)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=100,
eval_strategy=IntervalStrategy.STEPS,
eval_

## 16. Fine-Tune the Model

Start with the development subset. Once the pipeline is verified, increase the dataset size or train on the complete dataset.


In [16]:
train_result = trainer.train()
train_result

Step,Training Loss,Validation Loss
100,0.458786,0.485405
200,0.252259,0.411990
250,0.269653,0.414100


TrainOutput(global_step=250, training_loss=0.5019153327941894, metrics={'train_runtime': 2956.2496, 'train_samples_per_second': 0.677, 'train_steps_per_second': 0.085, 'total_flos': 2.3026143854592e+16, 'train_loss': 0.5019153327941894, 'epoch': 2.0})

## 17. Evaluate on the Validation Set


In [17]:
eval_result = trainer.evaluate()
eval_result

Training Loss,Validation Loss,Step
0.269653,0.414100,250


{'eval_loss': 0.4140995144844055}

## 18. Save the LoRA Adapter


In [18]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved to:", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))

Saved to: /kaggle/working/phi3_skill_lora
['adapter_model.safetensors', 'adapter_config.json', 'checkpoint-200', 'README.md', 'checkpoint-250', 'tokenizer_config.json', 'tokenizer.json', 'chat_template.jinja']


## 19. Export the Adapter as a ZIP Archive


In [19]:
ZIP_PATH = "/kaggle/working/phi3_skill_lora_adapter_v2"

shutil.make_archive(
    ZIP_PATH,
    "zip",
    OUTPUT_DIR,
)

print("Created:", ZIP_PATH + ".zip")
print("Please download the file(s) from the Output section of the Kaggle Notebook.")

Created: /kaggle/working/phi3_skill_lora_adapter_v2.zip
Please download the file(s) from the Output section of the Kaggle Notebook.


## 20. Run a Simple Inference Example


In [20]:
def predict_skill(title, description, max_new_tokens=12):
    model.eval()

    messages = [
        {
            "role": "user",
            "content": build_user_prompt(title, description),
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[
        0,
        inputs["input_ids"].shape[1]:
    ]

    prediction = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return prediction.splitlines()[0].strip()


sample_row = test_small.iloc[0]

prediction = predict_skill(
    sample_row["title"],
    sample_row["description"],
)

print("Job title:", sample_row["title"])
print("True skill:", sample_row["skill_name"])
print("Predicted skill:", prediction)

Job title: LICENSED PRACTICAL NURSE (LPN) - YADKIN NURSING CARE CENTER
True skill: Health Care Provider
Predicted skill: Health Care Provider


## Notes

- Input data: `/kaggle/input`
- Output directory: `/kaggle/working`
- Download the generated adapter ZIP from the **Output** panel after training.
- You may also click **Save Version** to preserve all notebook outputs.
